In [ ]:
from dotenv import load_dotenv

from pronostico_demanda_electrica.baselines import (
    dividir_entrenamiento_prueba,
    naive_estacional,
)
from pronostico_demanda_electrica.features import agregar_variables
from pronostico_demanda_electrica.metrics import wape
from pronostico_demanda_electrica.models import pronosticar_ols
from pronostico_demanda_electrica.queries import leer_demanda_mensual
from pronostico_demanda_electrica.storage import crear_engine

load_dotenv()
mensual = leer_demanda_mensual(crear_engine())

In [2]:
sin_total = (
    mensual[mensual["sistema"] == "SIN"]
    .groupby(["mes", "dias"], as_index=False)["demanda_gwh"]
    .sum()
    .assign(region="SIN-TOTAL")
)
sin_total.shape

(56, 4)

In [6]:
datos_sin = agregar_variables(sin_total)
entrenamiento_sin, prueba_sin = dividir_entrenamiento_prueba(datos_sin)
directo = pronosticar_ols(
    entrenamiento_sin.dropna(subset=["log_crecimiento"]), prueba_sin
)
naive_sin = naive_estacional(entrenamiento_sin, prueba_sin)

In [7]:
datos = agregar_variables(mensual)
entrenamiento, prueba = dividir_entrenamiento_prueba(datos)
por_region = pronosticar_ols(entrenamiento.dropna(subset=["log_crecimiento"]), prueba)

suma_regiones = (
    prueba.assign(pronostico=por_region)
    .query("sistema == 'SIN'")
    .groupby("mes")["pronostico"]
    .sum()
)

In [5]:
real = prueba_sin.set_index("mes")["demanda_gwh"]
resultados = {
    "naive_estacional": wape(real, naive_sin.set_axis(prueba_sin["mes"])),
    "ols_directo": wape(real, directo.set_axis(prueba_sin["mes"])),
    "ols_suma_regiones": wape(real, suma_regiones),
}
{nombre: round(valor, 2) for nombre, valor in resultados.items()}

{'naive_estacional': 2.55, 'ols_directo': 1.88, 'ols_suma_regiones': 1.87}